# 02 -- MetaPulsar (consistent) strategy on the real IPTA-DR2

The *consistent* combination strategy goes beyond a Frankenstein assembly by makign the timing models consistent across PTAs: every astrophysical parameter that lives in the merged components (`astrometry`, `spindown`, `binary`, `dispersion`) collapses to a **single fitted entry**, while detector-specific JUMPs / FD / DMX bins keep their per-PTA suffix. The result is a single `BasePulsar` whose timing model is astrophysically self-consistent.

1. **File and layout discovery** -- showcase MetaPulsar's regex-based directory walker, the canonical-name coordinate matcher, and `pta_summary`. These tools are how you go from "I have an IPTA release on disk" to "I have a `dict[pta_name -> list[par/tim entries]]` ready for `create_metapulsar`".
2. **Build a consistent MetaPulsar on real data** -- run the consistent combination on `J1853+1303` across EPTA dr2 + NANOGrav 9y, force a different reference PTA, and diff the rewritten consistent par files against the originals for manual check

## What does MetaPulsar do?

MetaPulsar builds an **astrophysically correct timing model in a reproducible way**, without ever touching the individual PTA data releases. Instead of merging `par`/`tim` files by hand into a single combined release and running one Tempo2/PINT/Vela.jl instance on it, MetaPulsar runs a separate fitter per PTA and only enforces that the *astrophysical* part of the timing model is consistent across them.

![Data combination vs. MetaPulsar](figures/metapulsar_vs_combination.png)

In principle, if each individual PTA's timing model is fully physical (i.e. all relevant astrophysical parameters are already included), the **consistent combination yields exactly the same likelihood** as the manual data combination. This equivalence is guaranteed by the fact that the timing model implemented by the timing packages is deterministic, and their procedure has been mathematically and numerically verified to be the same between timing packages.

Current differences between the two approaches come mainly from:

- the linearization procedure (non-linear timing is in-prep);
- ill-specified timing models;
- re-processed TOAs;
- changes in noise modeling.

These are considered **out of scope** of the data-combination process philosophy that MetaPulsar adheres to: the goal is to keep the per-PTA releases untouched and only guarantee astrophysical consistency, not to redo the upstream timing or noise analyses.

## Step 0 -- Recover session state

We need `DATA_ROOT_STR` and `PULSAR_SUBSET` from `00_setup.ipynb`. If they are not in the IPython store, re-run `00_setup.ipynb`.

In [ ]:
import sys
import warnings
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

import loguru

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# As before the hack because the PR is not merged yet
# These calls won't be necessary once the PR is merged
def quiet_loguru(level: str = "WARNING") -> None:
    loguru.logger.remove()
    loguru.logger.add(sys.stdout, level=level)

quiet_loguru()

%store -r DATA_ROOT_STR
%store -r PULSAR_SUBSET

DATA_ROOT = Path(DATA_ROOT_STR)
TARGET = PULSAR_SUBSET[0]
print(f"DATA_ROOT      = {DATA_ROOT}")
print(f"PULSAR_SUBSET  = {PULSAR_SUBSET}")
print(f"TARGET pulsar  = {TARGET}")

## Step 1 -- Layout discovery and `pta_summary`

MetaPulsar ships with a regex-based **layout discoverer** that scans a directory tree, identifies the conventions a PTA used for laying out its `.par`/`.tim` files (where they live, what suffixes they carry, which `INCLUDE` paths the tim files reference), and turns that into a `Layout` object. `combine_layouts` merges several `Layout`s into one, and `discover_files` resolves the actual files on disk.

Each PTA in IPTA-DR2 uses a different on-disk convention (EPTA's flat-per-pulsar tree, NANOGrav's release-versioned directories, PPTA's combined dr1+dr2 layout). The regex discoverer copes with all three out-of-the-box.

This discovery service has been tested with all currently publicly released PTA data (NANOGrav, PPTA, EPTA, MPTA, InPTA)

In [ ]:
from metapulsar import (
    combine_layouts,
    discover_files,
    discover_layout,
    filter_file_data_by_pulsars,
    get_pulsar_names_from_file_data,
    pta_summary,
)

# This works on all real PTA data releases we have tested with (EPTA, PPTA, NANOGrav, MPTA, InPTA...)
epta_layout = discover_layout(str(DATA_ROOT / "EPTA_v2.2"), name="EPTA dr2", verbose=False)
ppta_layout = discover_layout(str(DATA_ROOT / "PPTA_dr1dr2"), name="PPTA dr1dr2", verbose=False)
nanograv_layout = discover_layout(str(DATA_ROOT / "NANOGrav_9y"), name="NANOGrav 9y", verbose=False)

combined_layout = combine_layouts(epta_layout, ppta_layout, nanograv_layout)
file_data = discover_files(combined_layout, verbose=True)
quiet_loguru()

print("Discovered PTAs and pulsar counts:")
for pta, files in file_data.items():
    print(f"  {pta:<15s} -> {len(files):>3d} pulsars")

`get_pulsar_names_from_file_data` does **coordinate-based** matching across PTAs: it parses each `.par` file's RAJ/DECJ (or ELONG/ELAT), normalises to a canonical name, and returns the de-duplicated list of pulsars that appear in *any* PTA. This is what you want when one PTA labels a pulsar `J1853+1303` and another spells the J2000 name slightly differently or uses a `B`-name.

In [ ]:
pulsar_names = get_pulsar_names_from_file_data(file_data)
print(f"Coordinate-matched pulsars across all PTAs: {len(pulsar_names)}")
print("First few:", pulsar_names[:8])

`pta_summary` is the check you can do to see whether all pulsars have been discovered

In [ ]:
quiet_loguru()

pta_summary(file_data)

## Step 2 -- Filter to `PULSAR_SUBSET` and focus on `TARGET`

`filter_file_data_by_pulsars` shrinks `file_data` to just the pulsars in `PULSAR_SUBSET`. We then narrow further to a single pulsar (`TARGET = J1853+1303`) so the heavy `create_metapulsar` build stays under a minute on a laptop.

In [ ]:
# These are the pulsars currently in the subset
PULSAR_SUBSET

In [ ]:
quiet_loguru()

filtered_data = filter_file_data_by_pulsars(file_data, PULSAR_SUBSET)

# Inspect the json
filtered_data

In [ ]:
# Narrow further to a single pulsar (TARGET) so create_metapulsar stays fast.
# `filter_file_data_by_pulsars` accepts either a single pulsar name or a list,
# and resolves J/B name aliases via coordinate-based matching.
single_pulsar_data = filter_file_data_by_pulsars(file_data, TARGET)

#print(f"PTAs available for {TARGET}: {list(single_pulsar_data.keys())}")
pta_summary(single_pulsar_data)

## Step 3 -- Build a consistent MetaPulsar (auto reference PTA)

When `reference_pta=None` (the default), the factory picks the PTA with the longest timespan. We pre-stage the *original* per-PTA `.par` files into `./parfiles/` (so we can diff them against the rewritten consistent ones in step 5), then build the consistent MetaPulsar with `parfile_output_dir="./parfiles"`.

The bulk of the time is `parameter_manager.make_parfiles_consistent()` (parsing every par file into PINT, copying mergeable parameters, re-emitting new consistent par files). With only EPTA + NANOGrav for `J1853+1303` and ~1.5k TOAs, this is ~30-60 s on a laptop.

In [ ]:
import shutil

from metapulsar import create_metapulsar


# Back up the original par files
parfiles_dir = Path("./parfiles").resolve()
parfiles_dir.mkdir(exist_ok=True)

for pta, files in single_pulsar_data.items():
    src = Path(files[0]["par"])
    dest = parfiles_dir / f"{TARGET}_original_{pta}.par"
    shutil.copy(src, dest)
    print(f"  copied original  {pta:<15s} -> {dest.name}")

quiet_loguru()

# Create the consistent MetaPulsar (output new par files with consistent timing models)
# The function create_all_metapulsars can be used for batch creation
mp_consistent = create_metapulsar(
    file_data=single_pulsar_data,
    combination_strategy="consistent",
    combine_components=["astrometry", "spindown", "binary", "dispersion"],
    add_dm_derivatives=True,
    parfile_output_dir=str(parfiles_dir),
)

print()
print(f"Name              : {mp_consistent.name}")
print(f"Strategy          : {mp_consistent.combination_strategy}")
print(f"PTAs combined     : {list(mp_consistent._pulsars.keys())}")
print(f"Reference PTA     : {list(mp_consistent._pulsars.keys())[0]}")
print(f"Components merged : {mp_consistent.combine_components}")
print(f"Total TOAs        : {len(mp_consistent.toas)}")
print(f"Fit parameters    : {len(mp_consistent.fitpars)}")

In [ ]:
# Check out the resulting pulsar object
from enterprise.pulsar import BasePulsar

type(mp_consistent), isinstance(mp_consistent, BasePulsar)

In [ ]:
dir(mp_consistent)

In [ ]:
# Post-fit timing residuals vs MJD, coloured by receiver backend (from TIM flags)

mjd = np.asarray(mp_consistent.stoas, dtype=float) / 86400.0
res_us = np.asarray(mp_consistent.residuals, dtype=float) * 1e6
err_us = np.asarray(mp_consistent.toaerrs, dtype=float) * 1e6
be = np.char.strip(np.asarray(mp_consistent.backend_flags, dtype=str))
labels = np.where(be == "", "(no backend flag)", be)
uniq = sorted(np.unique(labels))

with plt.style.context("dark_background"):
    fig, ax = plt.subplots(figsize=(11, 4), layout="constrained")
    cmap = mpl.colormaps["tab10"]
    n_c = cmap.N if hasattr(cmap, "N") else 10
    for i, name in enumerate(uniq):
        m = labels == name
        if not np.any(m):
            continue
        c = cmap(i % n_c)
        ax.errorbar(
            mjd[m],
            res_us[m],
            yerr=err_us[m],
            fmt=".",
            markersize=3,
            alpha=0.9,
            elinewidth=0.5,
            capsize=0,
            color=c,
            ecolor=c,
            label=name,
            linestyle="None",
        )
    ax.set_xlabel("MJD")
    ax.set_ylabel("Residual (μs)")
    ax.set_title(f"{mp_consistent.name} consistent combination")
    ax.grid(True, alpha=0.1, linestyle="-")
    ax.legend(loc="best", fontsize=8, framealpha=0.92)


## Step 4 -- Force a different reference PTA

The reference PTA contributes the *values* of every merged parameter -- it is therefore the only PTA whose `.par` is preserved verbatim. Forcing a different reference is mostly a sensitivity knob: a well-constrained pulsar should be statistically indistinguishable across reference choices. We rebuild the same pulsar with NANOGrav 9y as the reference (when present) and write to a separate `parfiles_ngref/` directory.

In [ ]:
# A rule to prefer NANOGrav 9y if it is present
FORCED_REF = (
    "NANOGrav 9y"
    if "NANOGrav 9y" in single_pulsar_data
    else next(iter(single_pulsar_data))
)

quiet_loguru()

# Create the consistent MetaPulsar (output new par files with consistent timing models)
mp_consistent_ngref = create_metapulsar(
    file_data=single_pulsar_data,
    combination_strategy="consistent",
    reference_pta=FORCED_REF,
    parfile_output_dir="./parfiles_ngref",
)

print(f"Forced reference PTA   : {FORCED_REF}")
print(f"PTAs (reference first) : {list(mp_consistent_ngref._pulsars.keys())}")
print(f"Fit parameter count    : {len(mp_consistent_ngref.fitpars)}")

## Step 5 -- Diff the original vs. consistent par files

Check using text editor

## Step 6 -- Visualize the design matrix: composite vs. consistent

To make the *consistent* strategy concrete, build the same pulsar a second time with `combination_strategy="composite"` and `spy` the two design matrices side-by-side. Both `Mmat`s are reordered by PTA -- using the per-TOA `pta` flag MetaPulsar attaches to every TOA -- so the per-PTA blocks stack vertically, the same convention notebook 01 used for the FrankenPulsar.

* **Composite (left)** -- block-diagonal: each PTA's timing-model parameters only touch its own rows. This is the FrankenStat shape from notebook 01, reproduced here through MetaPulsar's `combination_strategy="composite"`.
* **Consistent (right)** -- the merged components (`astrometry`, `spindown`, `binary`, `dispersion`) collapse into single columns that span *all* rows, while detector-specific parameters (JUMPs, FD, DMX, ...) keep the block-diagonal structure.

In [ ]:
# Create a composite MetaPulsar (FrankenStat), equal to the consistent one above
mp_composite = create_metapulsar(
    file_data=single_pulsar_data,
    combination_strategy="composite",
)

In [ ]:
# Reorder fit parameters for the side-by-side spy plot below:
#   - drop the (many) DMX columns from the composite Mmat
#   - put shared (un-suffixed) parameters first, then group by PTA suffix.
# A parameter is per-PTA iff its name ends with `_<pta_name>` for one of the PTAs in
# `single_pulsar_data`. The composite case has no shared parameters, so this just
# groups by PTA. The consistent case naturally splits into a shared block followed
# by per-PTA detector parameters (JUMPs, FD, ...).
PTA_NAMES = list(single_pulsar_data)

def pta_group(name: str) -> int:
    """0 if `name` is shared, k+1 if it ends with `_<PTA_NAMES[k]>`."""
    for k, pta in enumerate(PTA_NAMES):
        if name.endswith(f"_{pta}"):
            return k + 1
    return 0


def reorder_cols(fitpars, keep=lambda n: True):
    """Indices of `fitpars` that pass `keep`, shared first then grouped by PTA."""
    return sorted(
        (i for i, n in enumerate(fitpars) if keep(n)),
        key=lambda i: (pta_group(fitpars[i]), i),
    )

# Column ordering for visual
comp_cols = reorder_cols(mp_composite.fitpars, keep=lambda n: not n.startswith("DMX"))
cons_cols = reorder_cols(mp_consistent.fitpars)

In [ ]:
# Sort TOAs by PTA so the per-PTA blocks stack vertically
# (same convention as the FrankenPulsar in 01_frankenstat_composite.ipynb)
isort_composite = np.argsort(mp_composite.flags["pta"], kind="stable")
isort_consistent = np.argsort(mp_consistent.flags["pta"], kind="stable")

fig, (ax_comp, ax_cons) = plt.subplots(1, 2, figsize=(11, 5), sharey=False)

comp_labels = [mp_composite.fitpars[i] for i in comp_cols]
cons_labels = [mp_consistent.fitpars[i] for i in cons_cols]

ax_comp.spy(
    mp_composite.Mmat[np.ix_(isort_composite, comp_cols)] != 0,
    aspect="auto",
    markersize=1,
)
ax_comp.set_title(f"FrankenStat / composite  shape={mp_composite.Mmat.shape}")
ax_comp.set_xlabel(f"fit parameter ({len(comp_cols)} shown, DMX hidden)")
ax_comp.set_ylabel("TOA index (sorted by PTA)")
ax_comp.xaxis.tick_bottom()
ax_comp.set_xticks(np.arange(len(comp_labels)))
ax_comp.set_xticklabels(comp_labels, rotation=45, ha="right", fontsize=6)

ax_cons.spy(
    mp_consistent.Mmat[np.ix_(isort_consistent, cons_cols)] != 0,
    aspect="auto",
    markersize=1,
)
ax_cons.set_title(f"MetaPulsar / consistent  shape={mp_consistent.Mmat.shape}")
ax_cons.set_xlabel(
    f"fit parameter ({len(cons_cols)} total: shared first, then per-PTA)"
)
ax_cons.xaxis.tick_bottom()
ax_cons.set_xticks(np.arange(len(cons_labels)))
ax_cons.set_xticklabels(cons_labels, rotation=45, ha="right", fontsize=7)

fig.suptitle(f"Design matrix for {TARGET}: composite vs. consistent (rows sorted by PTA)")
fig.tight_layout()

## Step 7 -- Non-linear timing (upcoming)

> **Status:** in-prep an need help! The exact entry points may shift before merge, but the user-facing flow is expected to look like the snippet below.

So far this notebook (and notebook 01) has stayed firmly inside the *linear* regime: we build a single MetaPulsar, freeze the timing model at its reference point, and expose the design matrix `Mmat` so that downstream noise-modelling code (Enterprise, etc.) can solve the linear-in-`Mmat` problem. The reference point itself never moves, and there is no way to ask "what would the residuals look like if I nudged `PX` by 0.01?" without rebuilding everything from scratch.
The upcoming refactor lifts that restriction by making the consistent MetaPulsar *callable*: parameter updates are pushed through the underlying PINT and libstempo backends, and a single `form_residuals()` call recomputes residuals end-to-end -- with the full non-linear timing model, not just a Taylor expansion.

The resulting update makes MetaPulsar the one-stop data combination tool that allows full timing analysis without having to move between PINT<-->Tempo(2) anymore

### Target user-facing API

```python
from metapulsar import (
    create_all_metapulsars,
    discover_layout,
    discover_files,
    combine_layouts,
)

# Load data of two PTAs (automatic pattern discovery and matching)
file_data = discover_files(combine_layouts(
    discover_layout("./ipta-dr2/EPTA_v2.2",   name="EPTA dr2"),
    discover_layout("./ipta-dr2/NANOGrav_9y", name="NANOGrav 9y"),
))

# Create all consistent MetaPulsars in one shot
metapulsars = create_all_metapulsars(
    file_data,
    combination_strategy="consistent",
)

# B-name and J-name pulsars are matched by coordinates automatically
psr = metapulsars["B1855+09"]   # internally holds both PINT and libstempo views

# Update parameter values and re-form residuals (same API style as Vela.jl)
psr.update_parameters({"PX": 0.01})
psr.form_residuals()            # delegates to PINT and libstempo, then merges

# The familiar Enterprise-style accessors still work
# NOTE: May switch to Vela.jl API at this layer, and expose Enterprise/Discovery layer differently
psr.residuals, psr.toas, psr.freqs

# Model parameters keep the consistent naming convention introduced in step 3:
psr.fitpars
# -> ['RAJ', 'PB', 'FD1_NANOGrav 9y', 'JUMP1_EPTA dr2', ...]
```

#### Aim to make the resulting objects will work with Vela.jl analysis tools as well.
 - Idea: should work with `JUG` going forward also!


## This is a timing tool, not a PTA tool
 - The resulting astrophysical timing models are fully consistent
 - This should not be thought of as a PTA-layer tool
 - Allows full analysis of the *timing model*

## Need help
I (Rutger) have lots of ideas on how to move forward, and I believe combining data this way will prove to be a great asset in combining Tempo2/PINT heterogeneous datasets without manual work. But my time is limited, and collaboration on MetaPulsar development is more than welcome